In [1]:
import numpy as np
!pip install optuna scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 14.9 MB/s eta 0:00:00


In [2]:
# 1. Cargar datos
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# 2. Definir el modelo base
modelo = RandomForestClassifier(random_state=42)

In [3]:
# 3. Definir el espacio de búsqueda
#    A diferencia de Randomized Search, aquí definimos rangos continuos/discretos
#    que el algoritmo explorará de forma guiada, no puramente al azar
espacio_busqueda = {
    "n_estimators": Integer(50, 500),
    "max_depth": Integer(3, 20),
    "min_samples_split": Integer(2, 20),
    "max_features": Real(0.1, 1.0, prior="uniform"),
    "bootstrap": Categorical([True, False]),
}

In [4]:
# 4. Configurar la búsqueda bayesiana
bayes_search = BayesSearchCV(
    estimator=modelo,
    search_spaces=espacio_busqueda,
    n_iter=20,              # número de evaluaciones
    scoring="accuracy",
    cv=5,
    random_state=42
)

In [5]:
# 5. Ejecutar la optimización
bayes_search.fit(X_train, y_train)

BayesSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
              n_iter=20, random_state=42, scoring='accuracy',
              search_spaces={'bootstrap': Categorical(categories=(True, False), prior=None),
                             'max_depth': Integer(low=3, high=20, prior='uniform', transform='normalize'),
                             'max_features': Real(low=0.1, high=1.0, prior='uniform', transform='normalize'),
                             'min_samples_split': Integer(low=2, high=20, prior='uniform', transform='normalize'),
                             'n_estimators': Integer(low=50, high=500, prior='uniform', transform='normalize')})

In [6]:
# 6. Resultados
print("Mejores hiperparámetros encontrados:", bayes_search.best_params_)
print(f"\nMejor score (CV): {bayes_search.best_score_}")

Mejores hiperparámetros encontrados: OrderedDict({'bootstrap': False, 'max_depth': 19, 'max_features': 0.1, 'min_samples_split': 12, 'n_estimators': 50})

Mejor score (CV): 0.9624076607387142


In [7]:
# 7. Evaluar en test
mejor_modelo = bayes_search.best_estimator_
y_pred = mejor_modelo.predict(X_test)


In [8]:
mejor_modelo.score(X_test,y_test)

0.951048951048951